# A/B Testing: Statistical Significance Analysis

Опис проєкту
Етапи проєкту

# Етап 1. Завантаження та перевірка даних
    → Google Drive
    → pandas
    → df.info()
    → перевірка test / test_group

# Етап 2. Визначення метрик
    → metrics

# Етап 3. Підготовка даних
    → session_counts
    → numerators

# Етап 4. Розрахунок статистичної значущості
    → alpha
    → цикл metric
    → цикл test
    → conversion
    → z-test
    → p-value
    → significant
    → result

# Етап 5. Перевірка результатів
    → 16 результатів
    → results_df

# Етап 6. Експорт результатів
    → CSV

# Етап 7. Посилання та опис Tableau
    → Tableau dashboard

# Етап 8. Висновки

# Етап 1. Завантаження та перевірка даних

На першому етапі завантажуємо набір даних, який використовувався для побудови дашборду в проєкті **Create Your A/B Testing Tool**.

Після завантаження перевіряємо структуру датасету, назви колонок та значення тестів і груп.

In [ ]:
# Підключаємо Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Імпортуємо бібліотеку для роботи з даними
import pandas as pd

# Завантажуємо датасет
file_path = '/content/drive/MyDrive/bq-results-20260914-190049-1789413077085/databaseab.csv'
df = pd.read_csv(file_path)

# Переглядаємо перші рядки
df.head()

Mounted at /content/drive


,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-01,Slovakia,mobile,Europe,Paid Search,2,2,new account,1
1,2020-11-04,Bolivia,desktop,Americas,Direct,2,1,new account,1
2,2020-11-05,Lebanon,mobile,Asia,Direct,2,2,new account,1
3,2020-11-06,Oman,desktop,Asia,Paid Search,2,2,new account,1
4,2020-11-06,Dominican Republic,desktop,Americas,Paid Search,2,1,new account,1


In [ ]:
# Перевіряємо структуру датасету
df.info()

# Перевіряємо, які тести та групи присутні в даних
print("Тести:")
print(sorted(df['test'].unique()))

print("\nГрупи:")
print(sorted(df['test_group'].unique()))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   date        800996 non-null  object
 1   country     800996 non-null  object
 2   device      800996 non-null  object
 3   continent   800996 non-null  object
 4   channel     800996 non-null  object
 5   test        800996 non-null  int64 
 6   test_group  800996 non-null  int64 
 7   event_name  800996 non-null  object
 8   value       800996 non-null  int64 
dtypes: int64(3), object(6)
memory usage: 55.0+ MB
Тести:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Групи:
[np.int64(1), np.int64(2)]


> **Примітка:** Group 1 використовується як контрольна група (Control),
> Group 2 — як тестова група (Test).

# Етап 2. Визначення метрик

Для аналізу використовуємо чотири метрики, визначені в завданні.

Кожна метрика розраховується як відношення кількості відповідної події до кількості сесій:

**Metric = numerator / denominator**

Назви метрик та спосіб їх розрахунку задаємо статично.

Кількість метрик та тестів у подальшому алгоритмі не фіксуємо — розрахунок виконується за допомогою циклів.

In [ ]:
# Словник метрик:
# numerator — подія, яку рахуємо
# denominator — знаменник, відносно якого розраховується метрика

metrics = {
    'add_payment_info / session': {
        'numerator': 'add_payment_info',
        'denominator': 'session'
    },
    'add_shipping_info / session': {
        'numerator': 'add_shipping_info',
        'denominator': 'session'
    },
    'begin_checkout / session': {
        'numerator': 'begin_checkout',
        'denominator': 'session'
    },
    'new_accounts / session': {
        'numerator': 'new account',
        'denominator': 'session'
    }
}

metrics

{'add_payment_info / session': {'numerator': 'add_payment_info',
  'denominator': 'session'},
 'add_shipping_info / session': {'numerator': 'add_shipping_info',
  'denominator': 'session'},
 'begin_checkout / session': {'numerator': 'begin_checkout',
  'denominator': 'session'},
 'new_accounts / session': {'numerator': 'new account',
  'denominator': 'session'}}

# Етап 3. Підготовка даних

На цьому етапі окремо розраховуємо:

- кількість сесій для кожного тесту та групи;
- кількість подій для кожної з чотирьох метрик.

Group 1 використовуємо як контрольну групу (Control), а Group 2 — як тестову групу (Test).

Ці дані будуть використані для розрахунку конверсій та статистичної значущості.

In [ ]:
# Розраховуємо кількість сесій для кожного тесту та групи

session_counts = (
    df[df['event_name'] == 'session']
    .groupby(['test', 'test_group'])['value']
    .sum()
    .to_dict()
)

# Розраховуємо кількість подій для кожної метрики,
# тесту та групи

numerators = {}

for metric, info in metrics.items():
    numerator_event = info['numerator']

    numerators[metric] = (
        df[df['event_name'] == numerator_event]
        .groupby(['test', 'test_group'])['value']
        .sum()
        .to_dict()
    )

print("Кількість сесій:")
print(session_counts)

print("\nКількість подій за метриками:")
for metric, values in numerators.items():
    print(f"\n{metric}:")
    print(values)

Кількість сесій:
{(1, 1): 45362, (1, 2): 45193, (2, 1): 50637, (2, 2): 50244, (3, 1): 70047, (3, 2): 70439, (4, 1): 105079, (4, 2): 105141}

Кількість подій за метриками:

add_payment_info / session:
{(1, 1): 1988, (1, 2): 2229, (2, 1): 2344, (2, 2): 2409, (3, 1): 3623, (3, 2): 3697, (4, 1): 3731, (4, 2): 3601}

add_shipping_info / session:
{(1, 1): 3034, (1, 2): 3221, (2, 1): 3480, (2, 2): 3510, (3, 1): 5298, (3, 2): 5188, (4, 1): 5128, (4, 2): 4956}

begin_checkout / session:
{(1, 1): 3784, (1, 2): 4021, (2, 1): 4262, (2, 2): 4313, (3, 1): 9532, (3, 2): 9264, (4, 1): 12555, (4, 2): 12267}

new_accounts / session:
{(1, 1): 3823, (1, 2): 3681, (2, 1): 4165, (2, 2): 4184, (3, 1): 5856, (3, 2): 5822, (4, 1): 8984, (4, 2): 8687}


# Етап 4. Розрахунок статистичної значущості

Порівнюємо конверсію контрольної та тестової груп для кожної метрики та кожного тесту.

Для перевірки статистичної значущості використовуємо **two-proportion z-test**, оскільки порівнюємо дві пропорції.

Рівень статистичної значущості:

**α = 0.05**

Якщо `p-value < α`, результат вважаємо статистично значущим.

Додатково розраховуємо абсолютну та відносну зміну метрики та визначаємо напрямок статистично значущої зміни.

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

# Рівень статистичної значущості
alpha = 0.05

results = []

# Проходимо по всіх метриках
for metric in metrics:

    # Проходимо по всіх A/B-тестах
    for test in df['test'].unique():

        # Group 1 — Control
        control_conversions = numerators[metric].get((test, 1), 0)
        control_sessions = session_counts.get((test, 1), 0)

        # Group 2 — Test
        test_conversions = numerators[metric].get((test, 2), 0)
        test_sessions = session_counts.get((test, 2), 0)

        # Розраховуємо конверсію кожної групи
        control_rate = control_conversions / control_sessions
        test_rate = test_conversions / test_sessions

        # Перевіряємо статистичну значущість різниці
        # Напрямок z_stat: Test − Control
        z_stat, p_value = proportions_ztest(
            [test_conversions, control_conversions],
            [test_sessions, control_sessions]
        )

        # Абсолютна різниця між групами
        absolute_difference = test_rate - control_rate

        # Відносна зміна
        relative_change = absolute_difference / control_rate

        # Перевіряємо статистичну значущість
        significant = p_value < alpha

        # Визначаємо напрямок результату
        if not significant:
            result = 'Not significant'
        elif absolute_difference > 0:
            result = 'Significant increase'
        else:
            result = 'Significant decrease'

        # Зберігаємо результат
        results.append({
            'test': test,
            'metric': metric,
            'control_conversions': control_conversions,
            'test_conversions': test_conversions,
            'control_sessions': control_sessions,
            'test_sessions': test_sessions,
            'control_rate': control_rate,
            'test_rate': test_rate,
            'absolute_difference': absolute_difference,
            'relative_change': relative_change,
            'p_value': p_value,
            'z_stat': z_stat,
            'significant': significant,
            'result': result
        })

# Етап 5. Перевірка результатів

На цьому етапі перевіряємо сформовану таблицю результатів.

Для кожного тесту та кожної метрики маємо:

- кількість конверсій у Control та Test;
- кількість сесій у Control та Test;
- конверсію обох груп;
- абсолютну різницю;
- відносну зміну;
- p-value;
- ознаку статистичної значущості;
- напрямок зміни.

У результаті маємо 16 порівнянь: 4 тести × 4 метрики.

In [ ]:
# Створюємо фінальну таблицю результатів
results_df = pd.DataFrame(results)

# Сортуємо результати за тестом та метрикою
results_df = (
    results_df
    .sort_values(['test', 'metric'])
    .reset_index(drop=True)
)

# Переглядаємо результати
results_df

,test,metric,control_conversions,test_conversions,control_sessions,test_sessions,control_rate,test_rate,absolute_difference,relative_change,p_value,z_stat,significant,result
0,1,add_payment_info / session,1988,2229,45362,45193,0.043825,0.049322,0.005497,0.125420,0.000087,3.924884,True,Significant increase
1,1,add_shipping_info / session,3034,3221,45362,45193,0.066884,0.071272,0.004388,0.065605,0.009226,2.603571,True,Significant increase
2,1,begin_checkout / session,3784,4021,45362,45193,0.083418,0.088974,0.005556,0.066606,0.002894,2.978783,True,Significant increase
3,1,new_accounts / session,3823,3681,45362,45193,0.084278,0.081451,-0.002827,-0.033543,0.122859,-1.542883,False,Not significant
4,2,add_payment_info / session,2344,2409,50637,50244,0.046290,0.047946,0.001656,0.035769,0.214608,1.240994,False,Not significant
5,2,add_shipping_info / session,3480,3510,50637,50244,0.068724,0.069859,0.001135,0.016510,0.477979,0.709557,False,Not significant
6,2,begin_checkout / session,4262,4313,50637,50244,0.084168,0.085841,0.001673,0.019882,0.340642,0.952898,False,Not significant
7,2,new_accounts / session,4165,4184,50637,50244,0.082252,0.083274,0.001022,0.012419,0.556000,0.588793,False,Not significant
8,3,add_payment_info / session,3623,3697,70047,70439,0.051722,0.052485,0.000763,0.014746,0.520112,0.643172,False,Not significant
9,3,add_shipping_info / session,5298,5188,70047,70439,0.075635,0.073652,-0.001983,-0.026212,0.157442,-1.413727,False,Not significant


In [ ]:
# Перевіряємо кількість отриманих результатів
print(f"Кількість результатів: {len(results_df)}")

# Маємо отримати:
# 4 тести × 4 метрики = 16 результатів

Кількість результатів: 16


## Етап 6. Експорт результатів

Зберігаємо фінальну таблицю у CSV-файл, який буде використано
для візуалізації результатів статистичного аналізу в Tableau.

In [ ]:
# Зберігаємо результати у CSV
output_path = '/content/drive/MyDrive/ab_test_significance_results.csv'

results_df.to_csv(output_path, index=False)

print(f"Файл збережено: {output_path}")

Файл збережено: /content/drive/MyDrive/ab_test_significance_results.csv


## Етап 7. Посилання та пояснення логіки розрахунків

Результати статистичного аналізу зберігаються у CSV-файлі та використовуються
для побудови та оновлення дашборду в Tableau.

### Посилання

- **Tableau Dashboard:** [https://public.tableau.com/app/profile/daria.kiriukhina/viz/ABTESTSTATISTICALSIGNIFICANCE/ABTestStatisticalAnalysis?publish=yes]
- **CSV з результатами:** [https://drive.google.com/file/d/1EuSmrO6cYikvPAAx5wrUkplEpOuNHUfq/view?usp=sharing]
- **Colab Notebook:** [https://colab.research.google.com/drive/1UuNSFsoIxXbWQnQYmQtq74htcrDkLFbw?usp=sharing]

### Логіка Python-розрахунку

1. Для кожної метрики визначається кількість відповідних подій та кількість сесій.
2. Для кожного тесту Group 1 використовується як контрольна група, а Group 2 — як тестова.
3. Для обох груп розраховується конверсія:

   `conversion = кількість подій / кількість сесій`

4. За допомогою `proportions_ztest` порівнюються конверсії контрольної та тестової груп.
5. Отримане `p-value` порівнюється з рівнем значущості `α = 0.05`.
6. Якщо `p-value < 0.05`, різниця вважається статистично значущою.
7. Додатково визначається напрямок зміни:
   - `Significant increase` — статистично значуще зростання;
   - `Significant decrease` — статистично значуще зниження;
   - `Not significant` — статистично значущої різниці не виявлено.
8. Для кожного порівняння також розраховуються абсолютна та відносна зміни метрики.

Такий підхід дозволяє автоматично розрахувати результати для всіх тестів
і метрик без жорсткого задання їх кількості в коді.

## Етап 8. Фінальні висновки

У межах аналізу було досліджено 4 A/B-тести та 4 ключові конверсійні метрики:

add_payment_info / session;
add_shipping_info / session;
begin_checkout / session;
new_accounts / session.

Для кожної комбінації тесту та метрики було порівняно результати контрольної групи (Group 1) та тестової групи (Group 2).

Для визначення статистичної значущості різниці між конверсіями використовувався двовибірковий z-тест для пропорцій з рівнем значущості α = 0.05.

Якщо p-value < 0.05, різниця між групами вважається статистично значущою. Якщо p-value ≥ 0.05, статистично значущої різниці не виявлено.

Test 1

Test 1 показав найбільш позитивний результат серед усіх проведених тестів.

Статистично значуще покращилися одразу три метрики:

add_payment_info / session — зростання приблизно на 12.54%, p = 0.000087;
add_shipping_info / session — зростання приблизно на 6.56%, p = 0.009226;
begin_checkout / session — зростання приблизно на 6.66%, p = 0.002894.

Метрика new_accounts / session знизилася приблизно на 3.35%, однак ця зміна не є статистично значущою (p = 0.122859).

Аналітичний висновок:
Test 1 демонструє позитивний вплив на основні етапи воронки. Зростання трьох метрик є статистично значущим, при цьому статистично значущого погіршення new_accounts / session не виявлено. Тому саме Test 1 має найпереконливіший позитивний результат серед досліджених експериментів.

Test 2

Для Test 2 жодна з чотирьох метрик не показала статистично значущої різниці між контрольною та тестовою групами.

Отже, для цього тесту немає достатніх статистичних доказів того, що зміна тестової версії вплинула на досліджувані показники.

Аналітичний висновок:
Test 2 не продемонстрував статистично підтвердженого ефекту. Це не означає, що між групами взагалі немає різниці, а лише те, що на основі наявних даних її не вдалося підтвердити як статистично значущу.

Test 3

У Test 3 статистично значущу зміну показала лише метрика:

begin_checkout / session — зниження приблизно на 3.35%, p = 0.012026.

Інші три метрики не продемонстрували статистично значущих змін.

Аналітичний висновок:
Test 3 не показав загального позитивного ефекту. Навпаки, на одному з важливих етапів воронки — begin_checkout / session — було зафіксовано статистично значуще зниження конверсії. Тому результат Test 3 потребує додаткового аналізу причин такого зниження перед прийняттям рішення щодо впровадження тестової версії.

Test 4

Для Test 4 статистично значущими були дві негативні зміни:

begin_checkout / session — зниження приблизно на 2.35%, p = 0.045934;
new_accounts / session — зниження приблизно на 3.36%, p = 0.017527.

Для add_payment_info / session та add_shipping_info / session статистично значущих змін не виявлено.

Аналітичний висновок:
Test 4 має негативний результат за двома метриками. Особливо важливо, що статистично значуще знизилися як begin_checkout / session, так і new_accounts / session. Тому цей тест не демонструє переваг тестової версії та потребує додаткового дослідження причин негативного ефекту.

Порівняння результатів тестів

Якщо порівняти всі чотири експерименти, результати суттєво відрізняються.

Test 1 є безумовним лідером за результатами статистичного аналізу: три ключові метрики показали статистично значуще зростання, а значущого погіршення new_accounts / session не було.

Test 2 не показав статистично значущого ефекту за жодною з досліджуваних метрик.

Test 3 продемонстрував статистично значуще погіршення begin_checkout / session.

Test 4 продемонстрував статистично значуще погіршення двох метрик — begin_checkout / session та new_accounts / session.

Таким чином, за результатами проведеного аналізу Test 1 має найкращі статистичні результати, тоді як Test 3 і Test 4 мають ознаки негативного впливу тестових змін.

Загальний аналітичний висновок

Проведений A/B-аналіз показав, що ефект тестових змін залежить від конкретного експерименту та метрики. Аналіз усіх чотирьох показників дозволив отримати більш повну картину впливу тестових версій на різні етапи користувацької воронки.

Найбільш позитивний результат продемонстрував Test 1. У ньому статистично значуще зросли add_payment_info / session, add_shipping_info / session та begin_checkout / session. При цьому зниження new_accounts / session не було статистично значущим. Це свідчить про найбільш стабільний позитивний результат серед усіх чотирьох експериментів.

Test 2 не продемонстрував статистично значущого впливу на жодну з досліджуваних метрик, тому на основі цього аналізу немає достатніх підстав стверджувати, що тестова версія дала кращий результат.

Test 3 та Test 4 показали статистично значущі негативні зміни окремих метрик. Зокрема, в обох тестах знизилася конверсія begin_checkout / session, а в Test 4 також статистично значуще знизилася new_accounts / session.

Отже, за результатами статистичного аналізу найбільш перспективним є Test 1. Його результати можна розглядати як найбільш переконливі з точки зору позитивного впливу на досліджувані метрики. Водночас остаточне бізнес-рішення доцільно приймати з урахуванням мети конкретного експерименту, пріоритетності метрик та додаткового аналізу сегментів користувачів.

Важливо також зазначити, що статистична значущість не дорівнює автоматично бізнес-значущості. Навіть статистично значуще зростання показника необхідно оцінювати з точки зору його практичної цінності для бізнесу.

Підсумок

Test 1 → позитивний результат ✅
Test 2 → статистично значущого ефекту не виявлено ⚪
Test 3 → негативний результат за begin_checkout / session ❌
Test 4 → негативний результат за begin_checkout / session та new_accounts / session ❌

Загальний висновок: серед досліджених експериментів Test 1 показав найкращий результат і найбільше статистичних доказів позитивного впливу тестової версії.